# Open the ASCII exported file

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import xarray as xr
import pandas as pd
%matplotlib widget

def heatmap_interactive(_x, _y, _data, _title, _cmap='jet', _symlog=False, _linthresh=1.0,_loglog=False, _lines = False):
    fig = plt.figure(figsize=(8, 8))
    gs = gridspec.GridSpec(2, 2, width_ratios=[1, 0.5], height_ratios=[0.5, 1], hspace=0.2, wspace=0.2)
    ax_main = plt.subplot(gs[1, 0])
    main_plot = ax_main.pcolormesh(_x, _y, _data, cmap=_cmap)
    ax_main.set(ylabel='Delay / ps', xlabel='Wavelength / nm')
    # set mixed log-lin scale with threshold value linthresh
    if _loglog:
        ax_main.set_xscale('log')
        ax_main.set_yscale('log')
    elif _symlog:
        ax_main.set_yscale('symlog', linthresh=_linthresh)
    # set axis range to min and max values
    #ax_main.set_xlim(_x[0],_x[-1])
    ax_main.set_xlim(-1,_x[-1])
    ax_main.set_ylim(_y[0],_y[-1]) 

    if _lines:
        # Overlay contour lines
        levels = np.linspace(np.min(_data), np.max(_data), 12)
        contour = ax_main.contour(_x, _y, _data, levels=levels, colors='black', linewidths=0.5)
        ax_main.clabel(contour, fmt="%.0e", fontsize=8)

    ax_kin = plt.subplot(gs[0, 0])
    line_kin, = ax_kin.plot(_x,np.zeros(_x.shape))
    kin_zero_line, = ax_kin.plot([_x[0],_x[-1]],[0,0], color="0.6")
    ax_kin.set_xlim(_x[0],_x[-1])
    # Kinetics plot y-axis (intensity)
    if _loglog:
        ax_kin.set_yscale('linear')  # keep linear, unless you want to log this too
    elif _symlog:
        ax_kin.set_yscale('symlog', linthresh=_linthresh)

    ax_spec = plt.subplot(gs[1, 1])
    line_spec, = ax_spec.plot(np.zeros(_y.shape),_y)
    spec_zero_line, = ax_spec.plot([0,0],[_y[0],_y[-1]], color="0.6")
    ax_spec.set_ylim(_y[0],_y[-1])        
    
    # This lower bounds list is necessary because the blocks in the 2D-plot cover a certain range
    def create_lower_bounds(_value_list):
        result = np.empty_like(_value_list)
        #first lower bound is equal to the lowest value in the nm-list
        result[0] = _value_list[0]
        #example: lower bound for 100 ps is 97.5 ps if the value prior is 95 ps, and 75 ps if the value prior is 50 ps.
        for i in range(1,len(_value_list)):
            result[i] = (_value_list[i]+_value_list[i-1])/2
        return result    
    
    nm_lower_bounds = create_lower_bounds(_y)
    time_lower_bounds = create_lower_bounds(_x)
    
    def nm_to_index(_nm):
        return np.where(_nm > nm_lower_bounds)[0][-1]
    
    def time_to_index(_time):
        return np.where(_time > time_lower_bounds)[0][-1]
    
    def mouse_move(event):
        x = event.xdata
        y = event.ydata
        if x is not None and y is not None:
            if x>=_x[0] and x<=_x[-1] and y>=_y[0] and y<=_y[-1]:
                # update spectra slice and rescale
                new_spec = _data[:,time_to_index(x)]
                line_spec.set_xdata(new_spec)
                spec_bounds = ax_spec.get_ylim()
                spec_range = new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].max()-new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].min()
                ax_spec.set_xlim(new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].min()-0.1*spec_range,new_spec[(_y>=spec_bounds[0]) & (_y<=spec_bounds[1])].max()+0.1*spec_range)            

                # update kinetic slice and rescale
                new_kin = _data[nm_to_index(y),:]
                line_kin.set_ydata(new_kin)
                kin_bounds = ax_kin.get_xlim()  
                kin_range = new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].max()-new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].min()                
                ax_kin.set_ylim(new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].min()-0.1*kin_range,new_kin[(_x>=kin_bounds[0]) & (_x<=kin_bounds[1])].max()+0.1*kin_range)
                
                # redraw figure
                fig.canvas.draw_idle()
             
    fig.canvas.mpl_connect('motion_notify_event', mouse_move) 
    
    # find max absolute value of 2D data in the specified zoom mode of the plot
    def get_maxvalue(_xlim, _ylim, _xvals, _yvals, _data_array):
        y_filter = (_yvals>=_ylim[0]) & (_yvals<=_ylim[1])
        x_filter = (_xvals>=_xlim[0]) & (_xvals<=_xlim[1])
        
        if not np.all(y_filter == False) and not np.all(x_filter == False):
            return np.amax(np.abs(_data_array[y_filter][:,x_filter]))
        else:
            return 0
    
    def on_xlims_change(event_ax):
        ax_kin.set_xlim(event_ax.get_xlim())
        
        new_max = get_maxvalue(event_ax.get_xlim(),event_ax.get_ylim(),_x,_y,_data)
        if new_max > 0:
            main_plot.set_clim(vmin=-new_max, vmax=new_max)

    def on_ylims_change(event_ax):
        ax_spec.set_ylim(event_ax.get_ylim())
        
        new_max = get_maxvalue(event_ax.get_xlim(),event_ax.get_ylim(),_x,_y,_data)
        if new_max > 0:
            main_plot.set_clim(vmin=-new_max, vmax=new_max)        

    ax_main.callbacks.connect('xlim_changed', on_xlims_change)
    ax_main.callbacks.connect('ylim_changed', on_ylims_change)
    main_plot.set_clim(vmin=np.min(_data), vmax=np.max(_data))
    fig.colorbar(main_plot, ax=[ax_main, ax_spec], orientation='vertical', label='Intensity (a.u.)')
    plt.show(block=False)




# Open Ldm files

In [ ]:
# Load the .ldm file assuming it's space-separated and has no header
filename = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Ana files/Full_spectrum/LDA_Results/dataset_dpp_lda_01_086.ldm'
data = pd.read_csv(filename, delim_whitespace=True, header=None, names=['lifetime','wavelength', 'intensity'])

# Extract unique coordinates
wavelengths = np.sort(data['wavelength'].unique())
lifetimes = np.sort(data['lifetime'].unique())

# Pivot the data into a 2D grid: rows -> wavelength, columns -> lifetime
intensity_grid = data.pivot(index='wavelength', columns='lifetime', values='intensity').to_numpy()

# Create xarray DataArray
da = xr.DataArray(
    intensity_grid,
    coords={'wavelength': wavelengths, 'lifetime': lifetimes},
    dims=['wavelength', 'lifetime'],
    name='intensity'
)

# Convert to xarray Dataset
ds_dpp = xr.Dataset({'intensity': da})

# Display dataset
ds_dpp


In [ ]:
heatmap_interactive(ds_dpp.lifetime,ds_dpp.wavelength, ds_dpp['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_loglog=False,_cmap='seismic',_lines = True) #dpp

In [ ]:
heatmap_interactive(ds_dchtmp.lifetime,ds_dchtmp.wavelength, ds_dchtmp['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_loglog=False,_cmap='seismic',_lines = True) #dchtmp

In [ ]:
ds_

In [ ]:
heatmap_interactive(.lifetime,ds.wavelength, ds['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_loglog=False,_cmap='seismic',_lines = True) #dmp

In [ ]:
heatmap_interactive(ds.lifetime,ds.wavelength, ds['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_loglog=False,_cmap='seismic',_lines = True) #dipp

In [ ]:
upper_time = 0.250
lower_time = 0.05

section = ds.sel(lifetime = slice(lower_time,upper_time))
integral = section.integrate(coord='lifetime')

# Plot result vs wavelength
plt.figure(figsize=(12, 6))
plt.plot(integral['wavelength'], np.abs(integral['intensity']))
plt.xlabel('Wavelength (nm)')
plt.ylabel('Abs Integrated intensity (a.u.)')
plt.title(f'Integrated signal from {lower_time} to {upper_time} ps')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Open 3D data map

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr

# Step 1: Load file
filename =  '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Ana files/Full_spectrum/LDA_fit_results_ASCII/dataset_dpp_lda_01_073.fit_a_c'

df = pd.read_csv(filename, delim_whitespace=True, header=None, names=['lifetime', 'wavelength', 'intensity'])

trace_length = 187  # Corrected block size
num_traces = df.shape[0] // trace_length

# Extract coordinates
lifetimes = df['lifetime'].values[:trace_length]
wavelengths = df['wavelength'].values[::trace_length]

# Reshape the intensity array
intensity_values = df['intensity'].values.reshape((num_traces, trace_length))

# Create xarray DataArray
da = xr.DataArray(
    data=intensity_values,
    coords={'wavelength': wavelengths, 'lifetime': lifetimes},
    dims=['wavelength', 'lifetime'],
    name='intensity'
)
# Convert to xarray Dataset
dat = xr.Dataset({'intensity': da})



In [ ]:
heatmap_interactive(dat.lifetime, dat.wavelength, dat['intensity'].transpose('wavelength','lifetime'),'Averaged scan plot',_symlog=False)

# Import the CA map from Optimus and perform the chirp correction

In [ ]:
from scipy.interpolate import interp1d


# Provided Optimus parameters
lambda_c = 485  # nm
c0 = 0.296      # ps
dispersion_coeffs = [1.30, 1.21, -1.00]  # c1, c2, c3

# Example: your dataset
wavelengths = dat['wavelength'].values
time = dat['lifetime'].values
data = np.transpose(dat['intensity'].values)  # shape (n_time, n_wavelength)

# Step 1: Calculate c(λ) for each wavelength
delta = (wavelengths - lambda_c) / 100
c_lambda = c0 + sum(c * delta**(i + 1) for i, c in enumerate(dispersion_coeffs))  # shape (n_wavelength,)

# Step 2: Interpolate and apply the shift
data_corrected = []

for i, shift in enumerate(c_lambda):
    trace = data[:, i]
    f_interp = interp1d(time - shift, trace, bounds_error=False, fill_value=np.nan)
    trace_shifted = f_interp(time)
    data_corrected.append(trace_shifted)

# Step 3: Reassemble into xarray
data_corrected = np.column_stack(data_corrected)

data_dispersion_corrected = xr.DataArray(
    data_corrected,
    coords={"lifetime": time, "wavelength": wavelengths},
    dims=["lifetime", "wavelength"],
    name="data_dispersion_corrected"
)



In [ ]:

heatmap_interactive(data_dispersion_corrected.lifetime, data_dispersion_corrected.wavelength, data_dispersion_corrected.transpose('wavelength','lifetime'),'Averaged scan plot')

In [ ]:
Tahara_wv = [475,500,525 ,550	,575  ,600 ,625, 650 ,675 ,700 ,725]
Tahara_values_dmp = [335.5, 446.7, 257.8, 93.3, 22.2, 0, 6.67, 2.22, 2.22, 22.2, 0]

# Plot result vs wavelength
plt.figure(figsize=(12, 6))
plt.plot(Tahara_wv, Tahara_values_dmp)
plt.show()

In [ ]:
section = data_dispersion_corrected.sel(lifetime = slice(-0.2,0.2))
integral_val = [] #Initialization of the array containing the min position
for k,wv in enumerate(section.wavelength):
    position_min = section.sel(wavelength = wv).idxmin().item()
    if section.sel(wavelength = wv).min().item() < 0:
        integral_val.append(np.abs(section.sel(wavelength = wv, lifetime = slice(position_min-0.1,position_min+0.05)).integrate(coord='lifetime').item()))
    else: 
        integral_val.append(0)



# Plot result vs wavelength
plt.figure(figsize=(12, 6))
plt.plot(section.wavelength.values, integral_val)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Abs Integrated intensity (a.u.)')
plt.title(f'Integrated early time signal - dpp')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# XY ASCII map

In [205]:
import numpy as np
import xarray as xr

# Path to your file
file_path = '/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Ana files/Full_spectrum/LDA_data_result_ASCII/dataset_dchtmp_replicate_lda_01_070.dst_a_c'

# Load the raw data
raw_data = np.loadtxt(file_path)

# Extract 1D array of wavelengths from the first row (odd indices)
wavelengths = raw_data[0, 1::2]  # Shape: (241,)

# Extract all time columns (even indices starting from 0), starting from second row
time_2d = raw_data[1:, 0::2]     # Shape: (187, 241)

# Extract all intensity values (odd indices starting from 1), starting from second row
intensity_2d = raw_data[1:, 1::2]  # Shape: (187, 241)

# Check shape consistency
assert time_2d.shape == intensity_2d.shape, "Time and intensity dimensions do not match"

# Create the xarray.Dataset
ds = xr.Dataset(
    data_vars={
        "intensity": (("scan", "wavelength"), intensity_2d),
        "time": (("scan", "wavelength"), time_2d)
    },
    coords={
        "wavelength": ("wavelength", wavelengths),
        "scan": np.arange(intensity_2d.shape[0])
    }
)

# Optional: print a quick summary
print(ds)


<xarray.Dataset> Size: 724kB
Dimensions:     (scan: 187, wavelength: 241)
Coordinates:
  * wavelength  (wavelength) float64 2kB 464.9 465.1 465.4 ... 519.6 519.9 520.1
  * scan        (scan) int64 1kB 0 1 2 3 4 5 6 7 ... 180 181 182 183 184 185 186
Data variables:
    intensity   (scan, wavelength) float64 361kB -4e-05 1e-05 ... 0.00561
    time        (scan, wavelength) float64 361kB -1.103 -1.104 ... 9.138 9.134


In [206]:
# Define a common time axis (e.g., 200 points between min and max)
common_time = np.linspace(ds.time.min().item(), ds.time.max().item(), 200)

# Interpolate intensity onto common time grid per wavelength
interpolated = np.empty((len(common_time), len(ds.wavelength)))

for j, wl in enumerate(ds.wavelength.values):
    # One time trace for this wavelength
    t = ds.time[:, j].values
    y = ds.intensity[:, j].values

    # Interpolate, handle potential issues with duplicate time values
    try:
        f = interp1d(t, y, kind='linear', bounds_error=False, fill_value=np.nan)
        interpolated[:, j] = f(common_time)
    except Exception as e:
        interpolated[:, j] = np.nan


#Create an xarray to manipulate the data (much easier)
dataset = xr.Dataset(
    {
        "data": (["time","spectral",], interpolated)
    },
    coords={
        "time": common_time,
        "spectral": ds.wavelength,
    }
)

# Print the dataset
print(dataset)


<xarray.Dataset> Size: 391kB
Dimensions:     (time: 200, spectral: 241, wavelength: 241)
Coordinates:
  * wavelength  (wavelength) float64 2kB 464.9 465.1 465.4 ... 519.6 519.9 520.1
  * time        (time) float64 2kB -1.866 -1.807 -1.748 ... 9.779 9.838 9.897
    spectral    (wavelength) float64 2kB 464.9 465.1 465.4 ... 519.6 519.9 520.1
Data variables:
    data        (time, spectral) float64 386kB nan nan nan nan ... nan nan nan


In [207]:
# Define the filename
filename = 'Cu(dchtmp)_afterLDA.ana'

# Open the file for writing
with open(filename, 'w') as f:
    # Write the header
    f.write("%FILENAME={}\n".format(filename))
    f.write("%DATATYPE=TAVIS\n")
    f.write("%NUMBERSCANS=1\n")
    f.write("%TIMESCALE=ps\n")

    # Write the time list
    f.write("%TIMELIST={}\n".format(" ".join(f"{value:.2f}" for value in dataset.time.values)))
    
    # Write the wavelength list
    f.write("%WAVELENGTHLIST={}\n".format(" ".join(f"{value:.2f}" for value in dataset.spectral.values)))
    
    # Write the intensity matrix
    intensity_matrix = dataset.data.values  # Get the intensity values
    i = 0
    for row in intensity_matrix:
        if i == 0:
            f.write("%INTENSITYMATRIX=\n {}\n".format(" ".join(f"{value:.5f}" for value in row)))
            i = i+1
        else:
            f.write("{}\n".format(" ".join(f"{value:.5f}" for value in row)))

print(f"File '{filename}' has been created successfully.")

File 'Cu(dchtmp)_afterLDA.ana' has been created successfully.
